# 프롬프트 엔지니어링 기초 1 (2026-03-24)

## 학습 목표
- 프롬프트가 무엇이고 왜 중요한지 이해한다
- 좋은 프롬프트의 4가지 구성 요소(지시 / 컨텍스트 / 입력 / 출력 형식)를 익힌다
- 역할 부여(Role Prompting), 단계별 지시(Step-by-step), CoT(Chain-of-Thought) 개념을 실습으로 체득한다
- `RoleBasedAssistant` 클래스로 멀티 에이전트 맛보기까지 구현한다

## 핵심 비유
> LLM은 **초능력 있는 신입사원**과 비슷해요.
> - 아는 건 많지만 **뭘 어디까지 해야 하는지 눈치는 약함**
> - 그래서 업무 지시를 할 때 "이거 잘 좀 해줘" 보다 "누구 대상, 분량, 포함/제외, 형식"을 구체적으로 말해줘야 함
> - 오늘 배울 프롬프트 엔지니어링은 결국 **업무 지시문 쓰는 스킬**입니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w3_memory_prompt_engineering/llm_260324_prompt_engineering_1.ipynb)

## 0. Colab 환경 설정
- Colab에서 처음 실행할 때만 패키지 설치
- `tenacity`는 API 호출 재시도용 (네트워크/토큰 이슈 대응)

In [ ]:
# Colab 환경에서 필요한 패키지를 설치합니다
!pip install langchain-text-splitters
!pip install langchain-openai
!pip install langchain_classic
!pip install tenacity

## 1. 프롬프트의 정의

**프롬프트(Prompt)** = LLM에 입력하는 **전체 텍스트**. 시스템 메시지 + 유저 메시지 + 대화 히스토리 모두 포함.

### 왜 프롬프트 엔지니어링이 필요할까?
GPT/Claude 같은 생성형 LLM은 **Transformer의 Decoder 구조**를 기반으로 하며, 내부적으로 "다음에 올 토큰의 확률 분포"를 따라 문장을 하나씩 생성합니다 (auto-regressive).
- 근본적으로 **랜덤성**에 기반 → 같은 질문에도 매번 답이 조금씩 달라짐
- 프롬프트를 구체적으로 쓰면 → 확률 분포가 **우리가 원하는 방향으로 쏠림** → 일관된 결과

### 좋은 프롬프트의 4요소
| 구성 요소 | 역할 |
|---|---|
| **지시 (Instruction)** | 무엇을 해야 하는지 |
| **컨텍스트 (Context)** | 어떤 상황/배경인지 |
| **입력 (Input)** | 처리 대상 데이터 |
| **출력 형식 (Output Format)** | 어떤 모양으로 응답할지 |

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 2. 나쁜 프롬프트 vs 좋은 프롬프트 체감해보기

### 나쁜 프롬프트의 문제
`"AI에 대해 알려줘"` 같은 모호한 질문을 LLM에 주면:
- **어떻게** 알려줄지? **어디부터 어디까지**? **얼만큼**? **누구한테**?
- 모델은 위 모든 것을 **자의적으로 해석** → 결국 학습 데이터에서 가장 흔한 형태로 답변
- → 길고, 장황하고, 우리 의도와 다른 결과

> **비유**: 식당에서 "맛있는 거 주세요" vs "매운 거 빼고, 2만원 이하, 해산물 위주로 추천해주세요"

In [ ]:
# 구성요소: 지시, context, input, output (format)
# 이 네 가지를 의식하면서 프롬프트를 써보자

In [ ]:
# 나쁜 예시: 지시만 있고 나머지 다 빠짐 → 매우 길고 모호한 답변이 나옴
bad_prompt = 'AI에 대해 알려줘'
llm.invoke([HumanMessage(content=bad_prompt)])

In [ ]:
# 좋은 예시: 4요소를 모두 갖춘 프롬프트
# [지시] [컨텍스트] [입력] [출력 형식] 을 명시적으로 구분
good_prompt = """
[지시] 아래 주제에 대해 초보자용 설명을 작성해 주세요.
[컨텍스트] 기업 신입사원 대상 AI 교육 자료입니다.
[입력] 대규모 언어모델 LLM의 작동 원리.
[출력 형식] 3문장 이내로, 비유를 하나 포함하세요.
"""
llm.invoke([HumanMessage(content=good_prompt)])

## 3. 프롬프트 빌더 함수 만들기

매번 f-string으로 프롬프트를 직접 쓰면 코드가 지저분해지니까, **재사용 가능한 함수**로 만들어두자.

> **비유**: 매번 손으로 편지 쓰는 대신 **편지 양식 템플릿**을 만들어두는 것

In [ ]:
def build_prompt(instructions, context="", input_data="", output_format=""):
    """
    4요소를 받아서 구조화된 프롬프트 문자열을 만들어주는 함수.
    context/input_data/output_format은 비어있으면 해당 블록을 생략함.
    """
    parts = []
    # 지시는 필수
    parts.append(f'[지시] {instructions}')
    # 선택 요소들은 값이 있을 때만 추가
    if context:
        parts.append(f'[컨텍스트] {context}')
    if input_data:
        parts.append(f'[입력] {input_data}')
    if output_format:
        parts.append(f'[출력 형식] {output_format}')
    # 줄바꿈으로 합쳐서 반환
    return '\n'.join(parts)

In [ ]:
# build_prompt 테스트: 고객 리뷰 감정 분석
prompt = build_prompt(
    instructions='다음 텍스트의 감정을 분석하세요.',
    context='고객 리뷰 분석 시스템입니다.',
    input_data='배송은 빨랐는데, 포장이 엉망이네요.',
    output_format='감정: [긍정/부정/혼합], 이유: [한 줄 설명]'
)
result = llm.invoke([HumanMessage(content=prompt)])

In [ ]:
# 결과 확인: 우리가 지정한 형식으로 응답하는지 확인
print(result)

### 실습: 뉴스 기사 요약기 만들기
`build_prompt` 함수를 이용해 뉴스 기사를 요약하는 프롬프트를 만들어봅니다.

In [ ]:
# build_prompt 함수를 이용해서 '뉴스 기사 요약기'를 만들어 보기
prompt = build_prompt(
    instructions='다음 기사를 분석 후 요약하세요.',
    context='기사 링크를 첨부합니다.',
    input_data="""
    양평군이 정부의 서울~양평 고속도로 사업 재개 발표(3월23일자 2면 보도)에 대해서 환영한다는 뜻을 밝히며 추후 노선결정 과정에서 강하IC 등 주민의견을 반영해 달라고 요청했다.
    군은 24일 양평군청 별관 4층 대회의실에서 '서울~양평 고속도로 재개 관련 기자회견'을 개최했다.
    전진선 군수는 \"지난 2023년 사업이 중단되며 우리 군민들이 겪은 충격과 고통은 이루 말할 수 없었다. 주민들은 사업 재개를 위한 집회와 현수막 게시, 대군민 서명운동을 통해 강하IC가 포함된 최선의 노선을 지속적으로 요구해 왔다\"며 \"서울~양평 고속도로 사업재개를 환영하며 본 사업이 군민의 의견을 적극 반영해 신속하게 추진되길 바란다\"고 입을 뗐다.
    """,
    output_format='타이틀: , 내용: 한줄로 요약하세요.'
)
result = llm.invoke([HumanMessage(content=prompt)])
print(result.content)

## 4. 명확한 프롬프트 = 랜덤성 줄이기

### 핵심 인사이트
> **명확할수록 랜덤성이 적어진다.**

`AI에 대해 알려줘`처럼 모호한 질문은 모델이 해석할 수 있는 범위가 너무 넓음.

### 명확함을 위한 4가지 체크리스트

| 항목 | 예시 |
|---|---|
| **수량** | "정확히 3문장", "5개 불릿", "200자 이내" |
| **대상** | "초등학생", "IT 기업 임원진", "프로그래밍 경험 없는 일반인" |
| **제외 조건** | "코드 예시 제외", "기술 용어 제외" |
| **품질 기준** | "친근한 대화체", "정확하고 간결하게" |

> **비유**: 배달 주문할 때 "맵기 보통, 양파 빼고, 새우는 추가, 포장으로" 식으로 구체화하는 것과 똑같음.

In [ ]:
# 명확함 비교 예시
# bad: 모델이 해석할 여지가 너무 많음 (어떻게? 누구한테? 얼마나?)
bad_prompt = '파이썬에 대해서 알려주세요'

# good: 수량/대상/포함/제외/문체를 모두 지정 → 예측 가능한 답변
good_prompt = '''파이썬 프로그래밍 언어에 대해 설명해 주세요.
조건:
- 대상 : 프로그래밍 경험이 없는 일반인
- 분량 : 정확히 3문장
- 포함 : 파이썬의 주요 용도 1가지
- 제외 : 코드 예시, 기술 용어
- 문체 : 친근한 대화체
'''

## 5. 역할 부여 (Role Prompting / Persona)

### 아이디어
`"당신은 10년차 시니어 개발자입니다"` 같이 역할(Persona)을 먼저 설정하면 응답 품질이 올라감.

### 왜 작동할까?
Transformer Decoder가 다음 토큰을 확률 분포로 예측할 때, **페르소나 설정은 그 분포를 특정 전문 영역으로 좁히는 역할**을 함.
- "당신은 수학자입니다" → 수학자들이 쓸 법한 단어·논리 분포 쪽으로 좁혀짐
- 그래서 더 정확하고 전문적인 추론이 나옴

### 논문 근거
- **Zero-shot Reasoning with Role-Play Prompting**: 역할 부여 시 성능 향상 입증
- **Expert Prompting (EgoRG)**: 전문가 페르소나를 구체적으로 줄수록 좋음 (동적으로 역할 생성 가능)
- **단, 모호한 페르소나("당신은 도움되는 어시스턴트입니다")는 오히려 방해가 될 수 있음** → 29페이지 짜리 논문의 실험 결과

> **비유**: 똑같은 사람이라도 "의사 가운" 입혀놓고 조언 받으면 의료적 답을, "변호사 정장" 입혀놓고 물어보면 법적 답을 주는 것과 같음.

In [ ]:
# 역할 부여 실험: 같은 질문도 역할에 따라 관점이 달라짐
question = "블록체인 기술이 금융 산업에 미치는 영향은?"
roles = {
    # 각 역할별로 구체적인 경력/소속/관점을 명시 (모호한 페르소나 X)
    "투자 전문가": "당신은 월스트리트에서 20년 경력의 투자 전문가입니다. 항상 투자 관점에서 생각하고, 고객의 이익을 최우선합니다. 구체적인 사례를 알려주세요.",
    "기술 개발자": "당신은 블록체인 코어 개발자입니다. 기술적 관점에서 설명하세요.",
    "규제 전문가": "당신은 금융감독원 출신의 규제 전문가입니다. 법적/제도적 관점에서 설명해 주세요."
}

# 각 역할로 동일한 질문을 던져서 답변 차이 확인
for role_name, role_msg in roles.items():
    print(f"roles: {role_name}")
    # SystemMessage에 역할을, HumanMessage에 질문을 담아 전달
    result = llm.invoke([
        SystemMessage(content=role_msg),
        HumanMessage(content=question)
    ]).content
    print(result[:300])  # 앞부분만 출력
    print()

## 6. 역할 기반 어시스턴트 클래스

매번 SystemMessage를 직접 관리하기 번거로우니 **`RoleBasedAssistant` 클래스**로 캡슐화.
- 대화 히스토리(`history`) 자동 관리
- `tenacity`로 API 재시도 로직 추가 (네트워크 이슈/None 응답 방지)

In [ ]:
# tenacity: 함수 실행이 실패하면 자동 재시도해주는 라이브러리
from tenacity import retry, wait_random_exponential, stop_after_attempt

class RoleBasedAssistant:
    """
    특정 역할(페르소나)을 가진 LLM 어시스턴트.
    - SystemMessage로 역할 고정
    - 대화 히스토리 자동 관리
    - API 호출 실패 시 자동 재시도
    """
    def __init__(self, role_name, system_prompt):
        # 역할명 (예: "CEO", "CTO")
        self.role_name = role_name
        self.system_prompt = system_prompt
        # 대화 시작 시 SystemMessage를 히스토리 맨 앞에 넣음
        self.history = [SystemMessage(content=system_prompt)]

    # 재시도 데코레이터: 최대 3번, 지수적 대기 (1~60초 랜덤)
    @retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(3))
    def ask(self, question):
        # 사용자 질문을 히스토리에 추가
        self.history.append(HumanMessage(content=question))
        # 전체 히스토리를 LLM에 전달 → 문맥 유지
        response = llm.invoke(self.history)
        answer = response.content
        # 간혹 LLM이 None을 반환하는 경우 → 재시도 트리거
        if answer is None:
            raise ValueError(f"Model for {self.role_name} returned None content")
        # AI 응답도 히스토리에 추가 (다음 턴에서 문맥 유지용)
        self.history.append(AIMessage(content=answer))
        return answer

    def reset(self):
        """대화 히스토리 초기화 (역할은 유지)"""
        self.history = [SystemMessage(content=self.system_prompt)]

In [ ]:
# 시니어 코드 리뷰어 에이전트 생성
reviewer = RoleBasedAssistant(
    role_name='시니어 코드 리뷰어',
    system_prompt="""
      당신은 10년 경력의 python 개발자이자 코드 리뷰어입니다.
      규칙:
      - 코드의 장단점을 균형있게 평가해 주세요.
      - 개선 제안은 항상 코드 예시와 함께해 주세요.
      - 주니어 개발자를 격려하는 톤을 유지해 주세요.
    """
)

In [ ]:
# 리뷰 대상 샘플 코드 (의도적으로 문제가 있는 코드)
sample_code = """
def calc(x):
    r = []
    for i in x:
        if i > 0:
            r.append(i*2)
    return r
"""
# reviewer.ask()로 코드 리뷰 요청
print(reviewer.ask(f"이 코드를 리뷰해주세요:\n{sample_code}"))

### 실습: 멀티 에이전트 맛보기 (CEO/CTO/CFO + 사회자)

여러 역할의 에이전트가 동일한 질문에 답하고, 사회자(Moderator)가 이를 종합하는 구조.

> **진짜 멀티에이전트 vs 맛보기 버전**
> - 진짜 멀티에이전트에 필요한 2가지: **① Tool 사용 (슬랙/결제/검색)**, **② Self-Reflection (상호 영향받아 자기반성/개선)**
> - 아래는 각자 독립 답변만 하니까 "멀티에이전트 시작점"이지 완전체는 아님

In [ ]:
# 단순한 버전의 멀티 에이전트 예시
# 1. 전문가 Assistant 생성 (각자 다른 관점 보유)
ceo_assistant = RoleBasedAssistant(
    role_name="CEO",
    system_prompt="""
    당신은 굴지의 IT 기업 CEO입니다. 새로운 AI 기술 도입 전략에 대해 비전, 시장 경쟁력, 투자 수익률 측면에서 설명해주세요.
    항상 회사의 장기적인 성장과 주주 가치 극대화를 최우선으로 생각하며 답변하세요.
    """
)

cto_assistant = RoleBasedAssistant(
    role_name="CTO",
    system_prompt="""
    당신은 IT 기업의 최고 기술 책임자(CTO)입니다. AI 도입 전략에 대해 기술적 타당성, 구현 가능성, 보안, 그리고 기존 시스템과의 통합 관점에서 설명해주세요.
    항상 최신 기술 트렌드와 안정적인 시스템 구축을 염두에 두고 답변하세요.
    """
)

cfo_assistant = RoleBasedAssistant(
    role_name="CFO",
    system_prompt="""
    당신은 IT 기업의 최고 재무 책임자(CFO)입니다. AI 도입 전략에 대해 예산, 비용 효율성, 재정적 리스크, 그리고 잠재적 수익성 관점에서 설명해주세요.
    항상 회사의 재정 건전성과 비용 대비 효과를 고려하며 답변하세요.
    """
)

# 2. 사회자 Assistant: 여러 전문가 의견을 종합
moderator_assistant = RoleBasedAssistant(
    role_name="사회자",
    system_prompt="""
    당신은 중요한 회의를 진행하는 노련한 사회자입니다. 주어진 전문가들의 의견을 명확하고 간결하게 요약하여 전달하세요.
    """
)

# 3. 질문 정의
question_ai_strategy = "우리 회사의 AI 도입 전략에 대해 의견을 제시해주세요."

# 4. 각 전문가에게 동일한 질문 → 각자 관점에서 답변 받기
ceo_response = ceo_assistant.ask(question_ai_strategy)
cto_response = cto_assistant.ask(question_ai_strategy)
cfo_response = cfo_assistant.ask(question_ai_strategy)

# 5. 사회자에게 3명의 답변을 하나의 메시지로 전달 → 종합 요청
combined_responses = f"""
[CEO 의견]
{ceo_response}

[CTO 의견]
{cto_response}

[CFO 의견]
{cfo_response}

위 전문가들의 'AI 도입 전략'에 대한 의견을 종합하여 간략하게 발표해주세요.
"""

moderator_summary = moderator_assistant.ask(combined_responses)

# 6. 사회자의 최종 종합 보고 출력
print(f"\n### 사회자 최종 보고 ###\n{moderator_summary}")

## 7. CoT (Chain-of-Thought) - 단계별 사고 유도

### 핵심 아이디어
복잡한 작업을 한 번에 시키지 말고, **단계별로 나눠서** 지시하자.

### 왜 효과가 있을까?
- **2022년 NeurIPS 논문**에서 입증: `"단계별로 생각해봐 (Let's think step by step)"` 문구 하나만 추가해도 **Zero-shot 추론 성능이 크게 향상**
- 이유: 모델은 토큰 제한이 있는데, 복잡한 문제를 바로 답하기엔 **연산량(토큰 양)이 부족**함
- 풀이 과정을 텍스트로 먼저 생성 → 추론에 필요한 연산을 분산 → 더 정확한 최종 답

### CoT와 Reasoning 모델의 관계
최근 나오는 **Reasoning 모델(o1, DeepSeek-R1 등)**은 CoT를 **자체적으로 확장한 구조**:
- 사용자에게 최종 답을 주기 전, 내부적으로 추론 체인을 여러 단계 돌림
- 그래서 Reasoning 모델은 `temperature`를 아예 막아두기도 함

> **비유**: 수학 문제 풀 때 "답: 42" 바로 쓰지 말고 **풀이 과정을 단계별로 써가며** 풀면 실수가 줄어드는 것과 같음.

In [ ]:
# tool (도구 사용): 슬랙 메시지 보내기, 결제(카드), 검색 등
# self-reflection: 답변 후 스스로 검토/반성
# CoT: Chain of Thought (사고의 연결고리)
# Reasoning: 추론 모델 (CoT를 내재화한 모델)

In [ ]:
# Before CoT: 한 번에 "분석하고 인사이트를 줘" → 구조가 들쑥날쑥
task_data = "2026년 우리 회사 매출 : Q1 50억, Q2 65억, Q3 45억, Q4 80억"
single_prompt = f'다음 데이터를 분석하고 인사이트를 제공하세요: {task_data}'
print(llm.invoke([HumanMessage(content=single_prompt)]).content)

In [ ]:
# After CoT: 단계별로 나눠서 지시 → 구조화된 답변
step_prompt = f"""다음 매출 데이터를 아래 단계에 따라 분석하세요.
데이터 : {task_data}

1단계 - 데이터 정리 : 분기별 매출을 표로 정리하세요
2단계 - 추세 분석 : 전분기 대비 증감률을 계산하세요
3단계 - 이상점 식별 : 눈에 띄는 변동이 있는 분기를 찾으세요
4단계 - 원인 추정 : 이상점의 가능한 원인을 2가지씩 제시하세요
5단계 - 제안 : 다음 분기 전략을 1가지 제안하세요

각 단계의 결과를 명확히 구분하여 출력하세요."""

# 단계별로 추출하면 원하는 답에 좀 더 가깝게 출력됨 → 함수화 가능 (아래)
result = llm.invoke([HumanMessage(content=step_prompt)]).content
print(result)

## 8. 재사용 가능한 CoT 빌더: `build_step_prompt`

매번 단계 프롬프트를 손으로 쓰지 말고 **함수화**하자.

In [ ]:
def build_step_prompt(task, data, steps):
    """
    단계별(Chain-of-Thought) 프롬프트를 자동 생성.
    - task: 전체 작업 목적 (예: "고객 리뷰 심층 분석")
    - data: 처리할 데이터
    - steps: [{'name': '...', 'instruction': '...'}] 형태의 단계 리스트
    """
    # 프롬프트 시작 부분 조립
    prompt_parts = [f'작업: {task}', f'\n데이터:\n{data}', '\n아래 단계를 순서대로 수행하세요:\n']

    # 각 단계를 번호 매겨서 추가
    for i, step in enumerate(steps, 1):
        prompt_parts.append(f"{i}단계 - {step['name']}: {step['instruction']}")

    # 출력 형식 지정 (## N단계 헤더로 구분)
    prompt_parts.append('\n각 단계의 결과를 "## N단계" 형식으로 구분하여 출력하세요.')
    return '\n'.join(prompt_parts)

In [ ]:
# 고객 리뷰 심층 분석에 적용
review = '배송은 정말 빨랐어요. 그런데 상품 색상이 사진과 좀 달라서 아쉬웠습니다.'

# 4단계로 분석 파이프라인 설계
steps = [
    {'name': '감정 식별', 'instruction': '리뷰에서 긍정, 부정 감정을 각각 추리세요.'},
    {'name': '주제 분류', 'instruction': '언급된 주제(배송, 품질, 가격 등)을 분류하세요.'},
    {'name': '점수 산정', 'instruction': '각 주제별 만족도를 1-5점으로 평가하세요.'},
    {'name': '종합 판단', 'instruction': '전체 만족도와 재구매 가능성을 판단하세요.'}
]

# 프롬프트 조립 후 확인
prompt = build_step_prompt(task='고객 리뷰 심층 분석', data=review, steps=steps)
print(prompt)

In [ ]:
# 실제 LLM 호출 → 단계별 결과 출력
response = llm.invoke([HumanMessage(content=prompt)]).content
print(response)

### 실습: 비즈니스 이메일 작성기

`build_step_prompt`로 두 가지 이메일을 생성:
1. 프로젝트 지연을 클라이언트에게 알리는 이메일 (사과/안내 톤)
2. 신규 파트너십 제안 이메일 (제안 톤)

In [ ]:
# build_step_prompt를 이용해서 비즈니스 이메일 작성
mail1 = '오늘 github actions에 크리티컬한 이슈가 발생하여 금일까지 처리하기로 했던 AA 프로젝트가 지연되었습니다.'
mail2 = '당신의 회사에 딱 맞는 서비스를 우리가 가지고 있습니다. BB 서비스의 고도화를 위해 파트너쉽을 맺기를 원합니다.'

# 이메일 작성 파이프라인: 요구사항 파악 → 핵심 메시지 → 유형 분석 → 구조 → 본문 → 검토
mailSteps = [
    {'name': '요구사항 파악', 'instruction': '발신자, 수신자, 상황의 긴급도, 목적을 명확히 파악하세요.'},
    {'name': '핵심 메시지 도출', 'instruction': '이메일에서 전달할 핵심 메시지를 한 문장으로 정리하세요.'},
    {'name': '유형 분석', 'instruction': '언급된 주제(제안, 사과, 컴플레인, 안내 등)을 분류하세요.'},
    {'name': '이메일 구조 설계', 'instruction': '인사->본론->요청사항->마무리 구조를 설계하세요.'},
    {'name': '본문 작성', 'instruction': '위 구조에 따라 완성된 이메일을 작성하세요. 유형에 따라 공손하지만 명확한 톤을 유지하세요.'},
    {'name': '최종 검토', 'instruction': '작성된 이메일의 톤, 오타, 논리 흐름을 검토하고 최종본을 출력하세요.'}
]

# 케이스 1: 프로젝트 지연 안내 이메일
prompt1 = build_step_prompt(task='프로젝트 지연을 클라이언트에게 알리는 이메일', data=mail1, steps=mailSteps)
response1 = llm.invoke([HumanMessage(content=prompt1)]).content
print(response1)
print()

# 케이스 2: 파트너십 제안 이메일
prompt2 = build_step_prompt(task='신규 파트너십 제안 이메일', data=mail2, steps=mailSteps)
response2 = llm.invoke([HumanMessage(content=prompt2)]).content
print(response2)

## 9. 오늘의 정리

| 기법 | 핵심 |
|---|---|
| **4요소 프롬프트** | 지시 / 컨텍스트 / 입력 / 출력 형식 |
| **명확성** | 수량·대상·제외조건·품질 기준을 구체화 → 랜덤성↓ |
| **역할 부여** | 구체적 전문가 페르소나로 확률 분포 좁히기 |
| **CoT** | 단계별 지시로 추론 정확도↑ |
| **RoleBasedAssistant** | 역할 + 대화 히스토리 + 재시도를 하나로 묶은 재사용 클래스 |
| **멀티 에이전트 맛보기** | 여러 역할 + 사회자 = 종합 의견 수렴 |

### 다음 시간(03-25) 예고
- **출력 형식 엔지니어링**: JSON/Markdown/YAML 등 구조화된 입출력
- **제약 조건 프롬프트**: `ConstrainedPrompt` 클래스
- **컨텍스트 주입**: Lost in the Middle 논문과 RAG의 관계
- **Zero-shot vs Few-shot**: 예시를 몇 개 줄 때 달라지는 성능

### 포맷이 LLM 응답에 미치는 영향? (Cliffhanger)
내일 실습으로 이어집니다.

In [ ]:
# 포맷이 LLM 응답에 미치는 영향?
# 내일(03-25) 이어집니다 → JSON/Markdown 등 구조화된 출력 실험